# MCP

In [1]:
import os

import chromadb
import dotenv
from agents import Agent, Runner, WebSearchTool, function_tool, trace
from agents.mcp import MCPServerStreamableHttp

dotenv.load_dotenv()

True

Let's set up our RAG database connection:

In [2]:
chroma_client = chromadb.PersistentClient(path="../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [3]:
# This is the same code as in the rag.ipynb notebook
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Set up WebSearchTool()

In [4]:
# # This is the same code as in the rag.ipynb notebook
# # @function_tool
# def web_lookup_tool(query: str, max_results: int = 3) -> str:
#     """
#     Tool function to search the web for any information that can be found online.

#     Args:
#         query: The food item to look up.
#         max_results: The maximum number of results to return.

#     Returns:
#         A string containing the nutrition information.
#     """

#     results = nutrition_db.query(query_texts=[query], n_results=max_results)

#     if not results["documents"][0]:
#         return f"No nutrition information found for: {query}"

#     # Format results for the agent
#     formatted_results = []
#     for i, doc in enumerate(results["documents"][0]):
#         metadata = results["metadatas"][0][i]
#         food_item = metadata["food_item"].title()
#         calories = metadata["calories_per_100g"]
#         category = metadata["food_category"].title()

#         formatted_results.append(
#             f"{food_item} ({category}): {calories} calories per 100g"
#         )

#     return "Nutrition Information:\n" + "\n".join(formatted_results)

Integrate EXA Search as an MCP:

In [ ]:
# Exa Search MCP code comes here:
exa_search_mcp = MCPServerStreamableHttp(
    name="Exa Search MCP",
    params={
        "url":f"https://mcp.exa.ai/mcp?exaApiKey={os.environ.get('EXA_API_KEY')}",
        "timeout": 30
    },
    client_session_timeout_seconds=30,
    cache_tools_list=True,
    max_retry_attempts=1
)

await exa_search_mcp.connect()

calorie_agent_with_search = Agent(
    name="Nutrition Assistant",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search the EXA web to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if necessary, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use Exa Search to find the exact recipe and ingredients.
    * Once you know the ingredients, use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 10 times.
    """,
    tools=[calorie_lookup_tool],
    mcp_servers=[exa_search_mcp]
)

In [ ]:

await exa_search_mcp.connect()

calorie_agent_with_web_search = Agent(
    name="Nutrition Assistant with WebSearchTool",
    instructions="""
    * You are a helpful nutrition assistant giving out calorie information.
    * You give concise answers.
    * You follow this workflow:
        0) First, use the calorie_lookup_tool to get the calorie information of the ingredients. But only use the result if it's explicitly for the food requested in the query.
        1) If you couldn't find the exact match for the food or you need to look up the ingredients, search the web to figure out the exact ingredients of the meal.
        Even if you have the calories in the web search response, you should still use the calorie_lookup_tool to get the calorie
        information of the ingredients to make sure the information you provide is consistent.
        2) Then, if necessary, use the calorie_lookup_tool to get the calorie information of the ingredients.
    * Even if you know the recipe of the meal, always use WebSearchTool to find the exact recipe and ingredients.
    * Once you know the ingredients, use the calorie_lookup_tool to get the calorie information of the individual ingredients.
    * If the query is about the meal, in your final output give a list of ingredients with their quantities and calories for a single serving. Also display the total calories.
    * Don't use the calorie_lookup_tool more than 10 times.
    """,
    tools=[calorie_lookup_tool, WebSearchTool()]
    # mcp_servers=[exa_search_mcp]
)

Reference query - shouldn't use ExaSearch:

In [7]:
with trace("Nutrition Assistant with MCP - Only uses calorie_lookup_tool"):
    result = await Runner.run(
        calorie_agent_with_search,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    - Estimated total for one banana (~118 g) + one apple (~182 g): about 200 kcal
    - Calories per 100 g: Banana 89 kcal; Apple 52 kcal
- 7 new item(s)
- 3 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


In [8]:
with trace("Nutrition Assistant with MCP "):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast? Look up the recipe ad combine the calories from each ingrediant"
    )
    print(result.final_output)

Do you want a typical Full English (sausages, bacon, eggs, tomatoes, mushrooms, beans, bread)? If yes, I can give a per-serving calorie breakdown using a standard recipe. Which version would you prefer (BBC/BBC Good Food example, or a specific restaurant-style version)?


In [12]:
with trace("Nutrition Assistant with WebSearchTool"):
    result = await Runner.run(
        calorie_agent_with_search, "How many calories are in an english breakfast? Look up the recipe ad combine the calories from each ingrediant"
    )
    print(result.final_output)

Estimated calories for a full English breakfast (using typical portions in the referenced recipe: 4 sausages, 4 slices back bacon, 4 eggs, 1 can baked beans, 1 cup mushrooms, 2 small tomatoes, 4 slices bread), per serving:

- Sausages (4 links, ~240 g): ~811 kcal
- Bacon (4 slices, ~112 g): ~456 kcal
- Eggs (4 large, fried): ~360 kcal
- Baked beans (1 can, ~420 g): ~395 kcal
- Mushrooms (1 cup cooked, ~150 g): ~33 kcal
- Tomatoes (2 small, ~100 g): ~18 kcal
- Bread (4 slices, white): ~286 kcal

Total: approximately 2,360 kcal per serving

Notes:
- Calories vary with exact weights and cooking method (fried vs. grilled, oil used, sausage type).
- If you use fewer components (e.g., no black pudding or fried bread), total will be lower.
